# Jitter Budget Calculation — PCIe Gen2 (Full-Rate, Half-Rate, Quarter-Rate)

Replicates the Matlab jitter-budgeting example from the CPPLL paper (Section: *A Design Framework for Jitter Budgeting in Serial Links*), comparing full-rate, half-rate, and quarter-rate architectures for the same link.

**Corrections already baked in (verified against the paper's equations):**
- Q-factor via `erfcinv` (matches $Q=\sqrt{2}\,\mathrm{erfc}^{-1}(2\cdot BER)$)
- Phase-noise-to-jitter integral includes the factor 2 (SSB $L(f)$ → double-sided integral)
- $DCD_{pp}$ referenced to $T_{CK}$ (the PLL/VCO output clock period), not the data UI
- $RJ_{pp,PLL}=2Q\sigma_{PLL}$ (peak-to-peak convention, consistent throughout)

**What changes between architectures:**
- $f_{VCO}$: full-rate = data rate; half-rate = data rate/2; quarter-rate = data rate/4
- $T_{CK}=1/f_{VCO}$ scales accordingly, which directly scales $DCD_{pp}$ and $Spurs_{pp}$
- $N$ (divider ratio) = $f_{VCO}/f_{ref}$ scales down, reducing reference-noise amplification
- $IQ_{pp}$ only applies to half-rate/quarter-rate (multi-phase) architectures


In [14]:
import math
from scipy.special import erfcinv
from si_prefix import si_format
from prettytable import PrettyTable

def eng(x, unit='s'):
    return si_format(x, precision=3) + unit


In [15]:
# ---- Common link-level parameters (same for all architectures) ----
DR = 5e9                      # data rate [bps]
UI = 1 / DR                   # data unit interval [s]
BER = 1e-12
Qfactor = math.sqrt(2) * erfcinv(2 * BER)   # Q-factor
TJ_max = 0.3 * UI              # PCIe Gen2 eye-mask budget [s]

fref = 100e6                   # reference clock frequency [Hz]
spurLevel = -40                # reference spur level [dBc]
DC_offset_perc = 0.5           # duty-cycle deviation from 50% [%]

print(f'Q-factor = {Qfactor:.4f}')
print(f'TJ budget = {eng(TJ_max)} (0.3 UI)')


Q-factor = 7.0345
TJ budget = 60.000 ps (0.3 UI)


In [16]:
def jitter_budget(fvco, label, has_IQ, Qoffset_deg=2):
    T_CK = 1 / fvco
    Ndiv = fvco / fref

    print(f'\n**** {label} ARCHITECTURE ****')
    print(f'f_VCO = {eng(fvco, "Hz")}, T_CK = {eng(T_CK)}, N = {Ndiv:.2f}')

    # --- Step 1: TJ decomposition (50/50 split) ---
    DJ_per = 0.5
    TJ_pp = TJ_max
    DJ_pp_budget = DJ_per * TJ_pp
    RJ_pp_budget = TJ_pp - DJ_pp_budget
    sigma_RJ_req = RJ_pp_budget / (2 * Qfactor)
    print(f'TJ_pp={eng(TJ_pp)}, DJ_pp(budget)={eng(DJ_pp_budget)}, '
          f'RJ_pp(budget)={eng(RJ_pp_budget)}, sigma_RJ,req={eng(sigma_RJ_req)}')

    # --- Step 2: PLL random-jitter target and VCO phase-noise requirement ---
    sigma_PLL = 1.2e-12        # target PLL RJ contribution [s, rms]
    f_low, f_high, f0 = 100e3, 1e9, 1e6
    # eq. 198 (with the 1/2 factor, consistent with the factor-2 correction in eq. 179)
    S_f0 = 0.5 * (f_low * f_high) / (f_high - f_low) * (2 * math.pi * fvco * sigma_PLL / f0) ** 2
    S_f0_dbc = 10 * math.log10(S_f0)
    print(f'Required VCO phase noise @{eng(f0,"Hz")} = {S_f0_dbc:.1f} dBc/Hz')

    # --- Step 3: Deterministic jitter contributions ---
    Spurs_pp = 10 ** (spurLevel / 20) / (math.pi * fvco)
    print(f'Spurs_pp = {eng(Spurs_pp)}')

    DC_offset_frac = DC_offset_perc / 100
    DCD_pp = 2 * DC_offset_frac * T_CK       # referenced to T_CK
    print(f'DCD_pp = {eng(DCD_pp)}  (DC deviation = {DC_offset_perc:.2f}%, referenced to T_CK)')

    IQ_pp = (Qoffset_deg / 360) * T_CK if has_IQ else 0
    if has_IQ:
        print(f'IQ_pp = {eng(IQ_pp)}  (quadrature error = {Qoffset_deg:.1f} deg)')

    DJ_pp_total = Spurs_pp + DCD_pp + IQ_pp
    print(f'DJ_pp = Spurs_pp + DCD_pp' + (' + IQ_pp' if has_IQ else '') + f' = {eng(DJ_pp_total)}')

    # --- Step 4: RJ contribution and budget closure ---
    RJ_pp_PLL = 2 * Qfactor * sigma_PLL      # factor 2 included
    TJ_pp_PLL = DJ_pp_total + RJ_pp_PLL
    margin = TJ_pp - TJ_pp_PLL
    print(f'RJ_pp,PLL = {eng(RJ_pp_PLL)}, TJ_pp,PLL = {eng(TJ_pp_PLL)}, margin = {eng(margin)}')

    return dict(fvco=fvco, T_CK=T_CK, Ndiv=Ndiv, S_f0_dbc=S_f0_dbc, Spurs_pp=Spurs_pp,
                DCD_pp=DCD_pp, IQ_pp=IQ_pp, DJ_pp=DJ_pp_total, RJ_pp_PLL=RJ_pp_PLL,
                TJ_pp_PLL=TJ_pp_PLL, margin=margin)


In [17]:
full    = jitter_budget(DR,     'FULL-RATE',    has_IQ=False)
half    = jitter_budget(DR / 2, 'HALF-RATE',    has_IQ=True)
quarter = jitter_budget(DR / 4, 'QUARTER-RATE', has_IQ=True)



**** FULL-RATE ARCHITECTURE ****
f_VCO = 5.000 GHz, T_CK = 200.000 ps, N = 50.00
TJ_pp=60.000 ps, DJ_pp(budget)=30.000 ps, RJ_pp(budget)=30.000 ps, sigma_RJ,req=2.132 ps
Required VCO phase noise @1.000 MHz = -101.5 dBc/Hz
Spurs_pp = 636.620 fs
DCD_pp = 2.000 ps  (DC deviation = 0.50%, referenced to T_CK)
DJ_pp = Spurs_pp + DCD_pp = 2.637 ps
RJ_pp,PLL = 16.883 ps, TJ_pp,PLL = 19.519 ps, margin = 40.481 ps

**** HALF-RATE ARCHITECTURE ****
f_VCO = 2.500 GHz, T_CK = 400.000 ps, N = 25.00
TJ_pp=60.000 ps, DJ_pp(budget)=30.000 ps, RJ_pp(budget)=30.000 ps, sigma_RJ,req=2.132 ps
Required VCO phase noise @1.000 MHz = -107.5 dBc/Hz
Spurs_pp = 1.273 ps
DCD_pp = 4.000 ps  (DC deviation = 0.50%, referenced to T_CK)
IQ_pp = 2.222 ps  (quadrature error = 2.0 deg)
DJ_pp = Spurs_pp + DCD_pp + IQ_pp = 7.495 ps
RJ_pp,PLL = 16.883 ps, TJ_pp,PLL = 24.378 ps, margin = 35.622 ps

**** QUARTER-RATE ARCHITECTURE ****
f_VCO = 1.250 GHz, T_CK = 800.000 ps, N = 12.50
TJ_pp=60.000 ps, DJ_pp(budget)=30.000 ps, RJ

## Summary table

Note: quarter-rate uses the same $f_{ref}=100~MHz$ as full/half-rate purely to keep the
three architectures comparable — $N=12.5$ is **not** a realizable integer divider ratio.
In a real design, $f_{ref}$ would typically be chosen (e.g. 125 MHz) so that $N$ lands on
an integer. Similarly, $IQ_{pp}$ for quarter-rate reuses the single-$\Delta\phi$ 4-phase
model as an illustrative extension — a real 8-phase quarter-rate design has more phase
pairs that can mismatch independently, so this should be treated as illustrative only.


In [18]:
rows = [
    ('f_VCO',       lambda d: eng(d['fvco'], 'Hz')),
    ('T_CK',        lambda d: eng(d['T_CK'])),
    ('N',           lambda d: f"{d['Ndiv']:.2f}"),
    ('S_phi,VCO(1MHz)', lambda d: f"{d['S_f0_dbc']:.1f} dBc/Hz"),
    ('Spurs_pp',    lambda d: eng(d['Spurs_pp'])),
    ('DCD_pp',      lambda d: eng(d['DCD_pp'])),
    ('IQ_pp',       lambda d: eng(d['IQ_pp']) if d['IQ_pp'] else '--'),
    ('DJ_pp',       lambda d: eng(d['DJ_pp'])),
    ('RJ_pp,PLL',   lambda d: eng(d['RJ_pp_PLL'])),
    ('TJ_pp,PLL',   lambda d: eng(d['TJ_pp_PLL'])),
    ('Margin',      lambda d: eng(d['margin'])),
]

table = PrettyTable()
table.field_names = ['Parameter', 'Full-Rate', 'Half-Rate', 'Quarter-Rate']
for name, fn in rows:
    table.add_row([name, fn(full), fn(half), fn(quarter)])

print(table)

+-----------------+---------------+---------------+---------------+
|    Parameter    |   Full-Rate   |   Half-Rate   |  Quarter-Rate |
+-----------------+---------------+---------------+---------------+
|      f_VCO      |   5.000 GHz   |   2.500 GHz   |   1.250 GHz   |
|       T_CK      |   200.000 ps  |   400.000 ps  |   800.000 ps  |
|        N        |     50.00     |     25.00     |     12.50     |
| S_phi,VCO(1MHz) | -101.5 dBc/Hz | -107.5 dBc/Hz | -113.5 dBc/Hz |
|     Spurs_pp    |   636.620 fs  |    1.273 ps   |    2.546 ps   |
|      DCD_pp     |    2.000 ps   |    4.000 ps   |    8.000 ps   |
|      IQ_pp      |       --      |    2.222 ps   |    4.444 ps   |
|      DJ_pp      |    2.637 ps   |    7.495 ps   |   14.991 ps   |
|    RJ_pp,PLL    |   16.883 ps   |   16.883 ps   |   16.883 ps   |
|    TJ_pp,PLL    |   19.519 ps   |   24.378 ps   |   31.874 ps   |
|      Margin     |   40.481 ps   |   35.622 ps   |   28.126 ps   |
+-----------------+---------------+-------------

## Measured Jitter Contributions
From the measured $\sigma_{pll}$ now we calculate $R_{J,pp,PLL}$ from the objective $BER$

In [20]:
sigma_PLL = 820e-15
RJ_pp_PLL = 2 * Qfactor * sigma_PLL

print('RJpp Contribution:'+si_format(float(RJ_pp_PLL))+'s')

RJpp Contribution:11.5 ps


## Figures of Merit

Same definitions used in the paper's comparison table (Section: *Validation of the Design Methodology*):

- $FOM_{Jitter} = 10\log_{10}\left[\left(\dfrac{\sigma_{rms}}{1s}\right)^2 \cdot \dfrac{P_{diss}}{1mW}\right]$ [dB]
- $FOM_{Power} = \dfrac{P_{diss}\,[mW]}{f_{CK}\,[GHz]}$
- $FOM_{Perf} = \sigma_{rms}^2 \cdot f_{CK}$  [$ps^2\,GHz$]

Enter your own $\sigma_{rms}$ (jitter) and $P_{diss}$ (power) below to compute all three.

In [21]:
# ---- Inputs: edit these with your own results ----
sigma_rms = 0.82e-12   # RMS jitter [s]
P_diss = 10.7e-3       # Power dissipation [W]
f_CK = 2.5e9           # Clock frequency [Hz]

# ---- Figures of Merit ----
FOM_Jitter = 10 * math.log10((sigma_rms / 1) ** 2 * (P_diss / 1e-3))
FOM_Power = (P_diss / 1e-3) / (f_CK / 1e9)
FOM_Perf = (sigma_rms / 1e-12) ** 2 * (f_CK / 1e9)   # sigma in ps, f_CK in GHz -> ps^2*GHz

table = PrettyTable()
table.field_names = ['Parameter', 'Value']
table.add_row(['sigma_rms', eng(sigma_rms)])
table.add_row(['P_diss', eng(P_diss, 'W')])
table.add_row(['f_CK', eng(f_CK, 'Hz')])
table.add_row(['FOM_Jitter', f'{FOM_Jitter:.1f} dB'])
table.add_row(['FOM_Power', f'{FOM_Power:.2f}'])
table.add_row(['FOM_Perf', f'{FOM_Perf:.2f} ps^2*GHz'])

print(table)

+------------+---------------+
| Parameter  |     Value     |
+------------+---------------+
| sigma_rms  |   820.000 fs  |
|   P_diss   |   10.700 mW   |
|    f_CK    |   2.500 GHz   |
| FOM_Jitter |   -231.4 dB   |
| FOM_Power  |      4.28     |
|  FOM_Perf  | 1.68 ps^2*GHz |
+------------+---------------+
